![Ростов-на-Дону: 8860 мест на карте, 922 пункта выдачи](attachment:rostov-hero.png)

# Одиннадцать задач про агентов

Практика недели 2. Лекция была про то, как модель зовёт инструменты и что из этой механики берёт на себя фреймворк. Здесь ты это делаешь руками — на OpenAI Agents SDK и на базе Ростова: 8860 мест из OpenStreetMap, из них 922 пункта выдачи заказов.

**Задачи независимы.** Застрял на одной — переходи к следующей, ничего не сломается. Зачёт — шесть задач из десяти; нулевая разминочная не в счёт.

**Что в ячейках.** Вид подписан в первой строке:

- `# ДАНО` — готовый код, читаешь и запускаешь.
- `# ЗАДАЧА N` — сюда пишешь ты, обычно две-пять строк.
- `# ПРОВЕРКА` — считает, сошлось или нет, и говорит, что именно не сошлось.

**Как запускать.** Две верхние ячейки — установка и настройка — обязательны и идут первыми: в них появляются `MODEL`, ключ и настроенный клиент, без которых не заработает ничего. Дальше задачи в любом порядке, но внутри задачи сначала `ДАНО`: там импорты и данные, на которых стоит остальное.

**Что понадобится.** Ключ к OpenAI-совместимому сервису — от самой OpenAI или от посредника. Все одиннадцать задач — это порядка сорока пяти запросов к модели на коротких промптах, то есть единицы рублей на недорогой модели. Но ключ твой: следи за расходом сам.

In [ ]:
# ДАНО: ставим фреймворк и забираем модули практики
!pip install -q openai-agents
!git clone -q --depth 1 https://github.com/myurushkin/ai_course_mmcs.git agentic-ai-course
%cd agentic-ai-course/week2/practice

## Настройка: куда SDK идёт по умолчанию и почему это не туда

`Agent` и `Runner` по умолчанию идут в OpenAI: в Responses API, в их же трассировку и на свою модель по умолчанию. Если ключ от посредника, ни одно из трёх умолчаний не подходит — а фреймворк об этом не предупреждает, он просто молча ходит не туда.

**Выбери сервис в первой строке ячейки.** С ключом OpenAI хватает `OPENAI_API_KEY`: клиент, протокол и трассировка уже настроены, и прогоны будут видны в панели OpenAI. С ключом посредника нужны ещё три вещи: свой клиент с `base_url`, переключение на `chat/completions` и выключенная трассировка — чужой ключ в неё всё равно не пройдёт.

Модель в обоих случаях задаётся переменной `OPENAI_DEFAULT_MODEL`: имя `MODEL` берите из каталога своего сервиса.

**Ещё одно.** Во всех примерах документации агент запускается через `Runner.run_sync(...)`. В ноутбуке это не сработает: здесь уже крутится цикл событий, и SDK честно отказывается — «cannot be called when an event loop is already running». Поэтому везде дальше `await Runner.run(...)`. В обычном скрипте `await` вне `async def` — синтаксическая ошибка, а в ячейке ноутбука так можно: IPython сам оборачивает её код в корутину.

In [ ]:
# ДАНО: ключ и настройка SDK. Работает и с ключом OpenAI, и с ключом посредника
import getpass, os
from openai import AsyncOpenAI
from agents import set_default_openai_api, set_default_openai_client, set_tracing_disabled

SERVICE = "openrouter"        # "openai" — если ключ от самой OpenAI
MODEL = "gpt-5-mini"        # имя модели берётся из каталога твоего сервиса

os.environ["OPENAI_API_KEY"] = getpass.getpass("Ключ: ")
os.environ["OPENAI_DEFAULT_MODEL"] = MODEL   # без этого SDK возьмёт свою модель по умолчанию

if SERVICE == "openai":
    BASE_URL = None                          # клиент по умолчанию и так идёт в OpenAI
else:
    BASE_URL = "https://api.aitunnel.ru/v1"
    set_default_openai_api("chat_completions")   # по умолчанию SDK идёт в Responses API
    set_default_openai_client(AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"],
                                          base_url=BASE_URL))

# трассировка прогонов уходит в OpenAI: со своим ключом это полезно, с чужим — бессмысленно
set_tracing_disabled(SERVICE != "openai")
print("настроено:", MODEL, "через", BASE_URL or "api.openai.com")

## Задача 0. Разминка: один вызов без фреймворка · 5 минут

Прежде чем звать `Runner`, посмотри на то, поверх чего он работает. Вызов инструмента — это не действие модели, а участок её ответа: имя функции и аргументы строкой. Выполняет вызов твой код.

**Что сделать.** Ничего писать не нужно: запусти ячейку и посмотри на вывод. Дальше всё это делает `Runner`.

In [ ]:
# ДАНО: тот же вызов, что в первой неделе, но теперь как точка отсчёта
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"], base_url=BASE_URL)
schema = {"type": "function", "function": {
    "name": "count_places",
    "description": "Сколько в городе мест такого вида. Возвращает одно число.",
    "parameters": {"type": "object", "required": ["kind"], "properties": {
        "kind": {"type": "string", "description": "вид места латиницей: pharmacy, cafe, outpost"}}}}}

answer = client.chat.completions.create(
    model=MODEL, tools=[schema],
    messages=[{"role": "user", "content": "Сколько в Ростове аптек?"}])

message = answer.choices[0].message
if message.tool_calls:
    call = message.tool_calls[0]
    print("модель попросила позвать:", call.function.name, call.function.arguments)
    print("функция при этом НЕ вызвана: её вызовет твой код")
else:
    print("модель решила ответить текстом, инструмент звать не стала:", message.content)
print("токенов:", answer.usage.prompt_tokens, "вход,", answer.usage.completion_tokens, "выход")

## Задача 1. Первый агент · 10 минут

Теперь то же самое делает фреймворк. Три готовые функции над базой города нужно превратить в инструменты и получить ответ на вопрос.

Подвох здесь один, и он тихий: у одной из трёх функций нет докстроки. Ошибки не будет, предупреждения тоже — просто агент будет промахиваться мимо неё. Модель видит только описание; пустое описание — это инструмент, о котором ей ничего не сказали.

**Про `function_tool`.** Он превращает обычную питоновскую функцию в инструмент: имя функции становится именем инструмента, аннотации типов — схемой аргументов, докстрока — описанием и описаниями полей. Писать его можно двумя способами, это одно и то же: `function_tool(count_places)` как вызов и `@function_tool` строкой над функцией — так будет в следующей задаче.

**Про `instructions`.** Это системное сообщение агента: текст, который SDK кладёт первым в каждый запрос к модели, перед вопросом пользователя. В нём пишут, чем агент занимается, чем ему пользоваться и чего не делать. Данные здесь одни на всю практику — база мест Ростова, к которой ходят инструменты; про неё и надо сказать модели, чтобы она не отвечала по памяти о городе вместо того, чтобы позвать инструмент.

**Что сделать.**

1. Запусти ячейку `ДАНО` и посмотри, у какой из трёх функций вместо описания написано `ДОКСТРОКИ НЕТ`.
2. Заполни `instructions`: агент отвечает на вопросы о Ростове по базе мест города, пользуясь инструментами, и не выдумывает числа. Своими словами — важно, что там сказано, а не как.
3. Той функции, у которой описания нет, задай его прямо при оборачивании: `function_tool(top_brands, description_override="…")`. Внутри — что инструмент возвращает и когда его звать. Файл `sdk_tasks.py` при этом не трогай.
4. Запусти ячейку задачи, потом ячейку `ПРОВЕРКА`.

In [ ]:
# ДАНО: три функции над базой города; посмотри, что у них в докстроках
import inspect
import sdk_tasks
from sdk_tasks import count_places, find_places, top_brands

for f in (count_places, find_places, top_brands):
    print(f.__name__, "→", (inspect.getdoc(f) or "ДОКСТРОКИ НЕТ").split("\n")[0])

In [ ]:
# ЗАДАЧА 1
from agents import Agent, Runner, function_tool

agent = Agent(
    name="Городская справочная",
    # instructions — системное сообщение: уходит модели перед каждым вопросом.
    # Напиши, что агент отвечает о Ростове по базе мест города через инструменты
    # и не выдумывает числа
    instructions="...",
    tools=[function_tool(count_places),
           function_tool(find_places),
           # у top_brands нет докстроки, поэтому описание задаётся здесь:
           # что инструмент возвращает и когда его звать
           function_tool(top_brands, description_override="...")])

result = await Runner.run(agent, "Сколько в Ростове круглосуточных аптек?")
print(result.final_output)

In [ ]:
# ПРОВЕРКА
sdk_tasks.check_first_agent(agent, result.final_output)

## Задача 2. Цепочка: результат одного инструмента — вход следующего · 12 минут

Вопрос «где ближайшая аптека и до скольки она работает» одним вызовом не решается. Нужны три подряд, и каждый следующий работает на том, что вернул предыдущий:

    адрес → координаты → id ближайшей аптеки → её часы работы

Между инструментами нет никакой передачи данных. Всё, что от первого инструмента доходит до второго, — это **текст его результата**, который модель прочитала. Если в тексте нет координат, взять их модели неоткуда, и цепочка обрывается. Поэтому формат результата — такой же интерфейс, как и описание.

**Что сделать.**

1. Запусти ячейку задачи как есть и посмотри на журнал вызовов внизу: цепочка оборвётся на первом же шаге.
2. Почини `return` в `geocode`: сейчас он возвращает адрес словами. Верни в строке ещё и координаты — так, чтобы их можно было прочитать глазами, например `lat=47.279, lon=39.749`.
3. Почини `return` в `nearest_place`: следующему инструменту нужен идентификатор места, а в строке его нет. Возьми `best['id']`.
4. Заполни `instructions`: пусть агент идёт по порядку — сначала адрес в координаты, потом ближайшее место по этим координатам, потом карточка этого места — и берёт координаты и `id` из ответов инструментов, а не из головы.
5. Прогони и запусти `ПРОВЕРКА`.

In [ ]:
# ДАНО: три инструмента и вопрос. Сами функции в базу уже ходят — смотри, что они ВОЗВРАЩАЮТ
from agents import Agent, Runner, function_tool
import sdk_tasks
from sdk_tasks import CHAIN_QUESTION, address_point, nearest, details

print(CHAIN_QUESTION)
print(address_point("Белорусская", "44"))

In [ ]:
# ЗАДАЧА 2
@function_tool
def geocode(street: str, housenumber: str) -> str:
    """Координаты дома по адресу."""
    place = address_point(street, housenumber)
    if place is None:
        return "такого адреса в базе нет"
    # верни строку так, чтобы следующий инструмент вычитал из неё координаты:
    # они лежат в place['lat'] и place['lon']
    return f"дом найден: {place['street']}, {place['housenumber']}"

@function_tool
def nearest_place(lat: float, lon: float, kind: str) -> str:
    """Ближайшее к точке место заданного вида: pharmacy, cafe, outpost."""
    found = nearest(lat, lon, kind, 1)
    if not found:
        return f"мест вида {kind} в базе нет"
    best = found[0]
    # верни строку так, чтобы из неё вычитался идентификатор места: best['id']
    return f"{best['name'] or 'без названия'} — {best['distance_m']} м"

# этот инструмент трогать не нужно: он последний в цепочке, из его результата
# ничего дальше не читают — сравни его с двумя верхними
@function_tool
def place_details(place_id: int) -> str:
    """Карточка места по его идентификатору: часы работы и телефон."""
    card = details(place_id)
    return f"{card['name']} · часы: {card['opening_hours']} · телефон: {card['phone']}"

agent = Agent(name="Городской помощник",
              # напиши в instructions порядок работы: сначала адрес в координаты,
              # потом по этим координатам ближайшее место, потом карточка места по его id.
              # И отдельно: координаты и id брать из ответов инструментов, а не придумывать
              instructions="...",
              tools=[geocode, nearest_place, place_details])

sdk_tasks.reset_log()
result = await Runner.run(agent, CHAIN_QUESTION)
print(result.final_output, "\n")
for call in sdk_tasks.CALLS:
    print("вызвано:", call)

In [ ]:
# ПРОВЕРКА
sdk_tasks.check_chain(result.final_output)

## Задача 3. Запрос объектом · 12 минут

До сих пор аргументы были плоскими: строка, число, флаг. Но параметром инструмента может быть и целый объект — модель соберёт его сама, а твой код получит готовый `PlaceQuery` и отдаст его в базу.

Ровно здесь видно, что даёт схема. Поле `kind: str` — это обещание строки и ничего больше: модель напишет «аптека», база вернёт пусто, ошибки не будет. Поле, у которого в схеме перечислены допустимые значения, выдумать нельзя: декодирование ограничено списком. Описанием можно попросить — схемой можно гарантировать.

**Что сделать.**

1. Запусти ячейку задачи как есть и посмотри на строки `→ {...}`: что агент положил в `kind` и нашла ли база хоть что-нибудь. Со свободной строкой туда попадает то русское слово, то латинское — как повезёт.
2. Замени тип поля `kind` со строки на `Kind` — готовый перечень видов мест, он уже импортирован из `sdk_tasks`. Получится `kind: Kind | None`.
3. Заполни `description=` у каждого поля: что это и когда его заполнять. Про `only_24_7` скажи прямо, что ставить `True` только на вопросы про круглосуточные.
4. Прогони и запусти `ПРОВЕРКА`: нужно 5 верно собранных запросов из 6.

In [ ]:
# ДАНО: функция, которая выполняет объект-запрос, и шесть вопросов с известным ответом
import sdk_tasks
from sdk_tasks import KINDS, QUERY_CASES, Kind, run_query
from agents import Agent, Runner, function_tool

print("виды мест в базе:", ", ".join(KINDS))
for question, expected in QUERY_CASES[:3]:
    print(f"  {question!r} → ждём {expected}")

In [ ]:
# ЗАДАЧА 3
from pydantic import BaseModel, Field

class PlaceQuery(BaseModel):
    """Запрос к базе мест города."""
    # у kind замени тип str на Kind — это готовый перечень видов мест из sdk_tasks
    kind: str | None = Field(None, description="...")     # впиши описание поля
    street: str | None = Field(None, description="...")   # впиши описание поля
    brand: str | None = Field(None, description="...")    # впиши описание поля
    only_24_7: bool = Field(False, description="...")     # впиши: когда ставить True
    limit: int = Field(5, description="сколько строк вернуть")

@function_tool
def search_places(q: PlaceQuery) -> str:
    """Поиск мест города по запросу."""
    return run_query(q)

agent = Agent(name="Поиск по городу",
              instructions="Отвечай по базе мест Ростова, пользуясь поиском.",
              tools=[search_places])

asked = []
for question, _ in QUERY_CASES:
    sdk_tasks.reset_log()
    await Runner.run(agent, question)
    asked.append(next((c["query"] for c in sdk_tasks.CALLS if c["name"] == "run_query"), {}))
    print(question, "→", asked[-1])

In [ ]:
# ПРОВЕРКА
sdk_tasks.check_query(asked)

## Задача 4. Описание решает · 12 минут

Два похожих инструмента: список мест и ближайшее к точке. Модель выбирает между ними по описаниям — больше у неё ничего нет. Восемь вопросов, у каждого известно, какой инструмент правильный; метрика — доля верного выбора.

Менять можно **только докстроки**. Выигрывает не красота, а сказанное прямо: что инструмент делает, когда его звать и когда звать не надо.

**Что сделать.**

1. Запусти ячейку задачи как есть и запомни число: с описаниями в одно слово выбор вырождается в угадывание.
2. Перепиши докстроку каждого из двух инструментов по одному шаблону: первая строка — что делает; дальше — когда звать; отдельной фразой — когда звать **не** надо и какой инструмент для этого есть; в конце — что вернёт.
3. Добавь секцию `Args:` с описанием каждого параметра — из неё фреймворк берёт описания полей схемы.
4. Прогоняй ячейку и смотри на число, пока не станет 0.9 и выше. Меняй только текст докстрок, код инструментов трогать нельзя.

In [ ]:
# ДАНО: восемь вопросов и прогонялка, считающая долю верного выбора
import sdk_tasks
from sdk_tasks import QUESTIONS, share_correct, find_places as db_find, nearest as db_nearest
from agents import Agent, Runner, function_tool

for question, want in QUESTIONS[:4]:
    print(f"  {question!r} → {want}")

In [ ]:
# ЗАДАЧА 4
@function_tool
def find_places(kind: str, street: str | None = None) -> str:
    # перепиши докстроку ниже: первая строка — что делает; дальше — когда звать;
    # отдельной фразой — когда звать НЕ надо и какой инструмент для этого есть;
    # в конце — что вернёт и секция Args с описанием kind и street
    """Поиск."""
    return db_find(kind, street)

@function_tool
def nearest_place(lat: float, lon: float, kind: str) -> str:
    # и здесь тоже: что делает · когда звать · когда звать НЕ надо · что вернёт · Args
    """Места."""
    found = db_nearest(lat, lon, kind, 1)
    if not found:
        return f"мест вида {kind} в базе нет"
    return f"{found[0]['name'] or 'без названия'} — {found[0]['distance_m']} м"

agent = Agent(name="Справочная", instructions="Отвечай по базе города.",
              tools=[find_places, nearest_place])

score = await share_correct(agent, Runner)
print("доля верного выбора:", score)

In [ ]:
# ПРОВЕРКА
sdk_tasks.check_share(score)

## Задача 5. Что нельзя выбрасывать · 10 минут

Модель не помнит ничего между вызовами: помнит история, которую ты отправляешь. `SQLiteSession` ведёт её за тебя.

Диалог из трёх реплик, и последняя осмысленна только вместе с первой: «а сколько из них круглосуточных?» — из каких? Прогони диалог дважды, с сессией и без, и посмотри, где он рассыпается.

**Что сделать.**

1. Первый цикл уже написан — он гоняет диалог без памяти, каждый вопрос с чистого листа.
2. Заведи сессию вместо многоточия: `SQLiteSession("любое-имя-разговора")`.
3. Во втором цикле передай её в запуск: `await Runner.run(agent, question, session=session)`.
4. Сравни два последних ответа и запусти `ПРОВЕРКА`.

In [ ]:
# ДАНО: диалог и агент
import sdk_tasks
from sdk_tasks import DIALOG, count_places, find_places
from agents import Agent, Runner, SQLiteSession, function_tool

agent = Agent(name="Справочная", instructions="Отвечай по базе города Ростова.",
              tools=[function_tool(count_places), function_tool(find_places)])
for line in DIALOG:
    print(" ·", line)

In [ ]:
# ЗАДАЧА 5
without_memory = []
for question in DIALOG:
    without_memory.append((await Runner.run(agent, question)).final_output)   # каждый раз с чистого листа

session = ...            # заведи сессию: SQLiteSession с любым именем разговора
with_memory = []
for question in DIALOG:
    # добавь в запуск аргумент session — тогда история диалога поедет вместе с вопросом
    with_memory.append((await Runner.run(agent, question)).final_output)

print("без памяти:", without_memory[-1])
print("с памятью: ", with_memory[-1])

In [ ]:
# ПРОВЕРКА
sdk_tasks.check_session(without_memory[-1], with_memory[-1])

## Задача 6. Предохранитель и цена · 10 минут

Инструмент, который на любой запрос вежливо просит уточнить фильтр. Агент уточняет, зовёт снова, получает то же самое — и так пока кто-нибудь его не остановит.

Чинить тут нечего: промпт ни при чём, инструмент просто не даёт продвинуться. Нужен предохранитель и счёт денег.

**Что сделать.**

1. Подставь в `turns` число шагов, после которого агента нужно остановить, — оно уходит в `max_turns`. Ставь столько, чтобы круг успел стать видимым, но деньги не утекли.
2. Посчитай цену второго, нормального прогона. Токены прогона лежат в `ok.context_wrapper.usage` — это объект, который SDK ведёт по ходу работы; в нём `input_tokens` и `output_tokens`. Перевести их в рубли по тарифу умеет готовая `cost_rub`.
3. Запусти `ПРОВЕРКА`.

In [ ]:
# ДАНО: упрямый инструмент и цена токенов в тарифе сервиса
import sdk_tasks
from sdk_tasks import PRICE_IN, PRICE_OUT, cost_rub, count_places, stubborn_tool
from agents import Agent, MaxTurnsExceeded, Runner, function_tool

stubborn = Agent(name="Упрямый поиск", instructions="Найди места по вопросу пользователя.",
                 tools=[function_tool(stubborn_tool)])
normal = Agent(name="Справочная", instructions="Отвечай по базе города.",
               tools=[function_tool(count_places)])
print("тариф:", PRICE_IN, "₽ за миллион входных,", PRICE_OUT, "₽ за миллион выходных")
print("числа примерные — подставь цены своего сервиса, если знаешь их")

In [ ]:
# ЗАДАЧА 6
stopped = False
turns = 0                        # впиши число: сколько шагов дать агенту, прежде чем остановить
try:
    await Runner.run(stubborn, "Найди места в городе", max_turns=turns)
except MaxTurnsExceeded:
    stopped = True
    print("упёрлись в лимит шагов")

ok = await Runner.run(normal, "Сколько в Ростове аптек?", max_turns=3)
price = 0.0                      # впиши цену прогона: токены лежат в ok.context_wrapper.usage,
                                 # перевести в рубли умеет cost_rub
print("прогон стоил", price, "₽")

In [ ]:
# ПРОВЕРКА
sdk_tasks.check_limit(stopped, turns, price)

## Задача 7. Форма, при которой нельзя соврать · 12 минут

`output_type` заставляет агента отдать объект вместо текста. Объект проходит валидацию — и всё равно бывает враньём: вот отчёт, где у места спрос 999 при нуле метров до ближайшего конкурента и ноль просмотренных кварталов.

Схема гарантирует форму, а не правду. Но часть вранья формой отсекается: добавь границы, при которых подложный отчёт не пройдёт. Осторожно: перекрутишь — отсечёшь и правдивый.

**Осторожно с многоточием.** В `Field(...)` многоточие — не наша заготовка, а способ pydantic сказать «поле обязательное». Его нужно оставить на месте, а ограничения дописать после запятой: `Field(..., ge=0, le=500)`.

**Что сделать.**

1. Посмотри на два отчёта в ячейке `ДАНО`: чем подложный отличается от настоящего.
2. Проставь в `Field(...)` границы: у `demand` — разумный диапазон через `ge=` и `le=`, у `competitor_distance_m` — почему ноль невозможен, у `checked_cells` — сколько кварталов минимум надо было посмотреть, чтобы выбор был выбором.
3. Запусти `ПРОВЕРКА`: подложный отчёт должен падать, настоящий — проходить. Оба гоняются одной проверкой, так что перекрученные границы видно сразу.

In [ ]:
# ДАНО: два отчёта, подложный и настоящий
import sdk_tasks
from sdk_tasks import FAKE_REPORT, TRUE_REPORT

print("подложный:", FAKE_REPORT)
print("настоящий:", TRUE_REPORT)

In [ ]:
# ЗАДАЧА 7
from pydantic import BaseModel, Field

class Place(BaseModel):
    lat: float
    lon: float
    demand: int = Field(...)                      # впиши границы спроса: ge= и le=
    competitor_distance_m: int = Field(...)       # впиши границу: нуля метров быть не может
    why: str

class Report(BaseModel):
    places: list[Place]
    checked_cells: int = Field(...)               # впиши границу: минимум просмотренных кварталов

In [ ]:
# ПРОВЕРКА
sdk_tasks.check_form(Report)

## Задача 8. Гардрейл · 10 минут

Агент послушно напишет стихотворение про Ростов — за твои деньги и в рабочее время. Отсеки постороннее до того, как за него заплатят.

По умолчанию гардрейл идёт параллельно агенту: проверка и работа стартуют вместе, и на срабатывании прогон отменяется. Чтобы проверка шла строго до агента — `run_in_parallel=False`.

**Как это устроено.** Гардрейл — обычная функция, её зовёт SDK. Ей передают три вещи: `ctx` — контекст прогона, `agent` — кого собираются запускать, `text` — что пришло от пользователя; в этой задаче нужен только третий. Вернуть надо `GuardrailFunctionOutput`, и решает в нём поле `tripwire_triggered`: `True` означает «стоп», SDK прерывает прогон исключением `InputGuardrailTripwireTriggered` — его и ловит цикл ниже. `output_info` никуда не влияет, это место для пояснения, почему сработало.

**Что сделать.**

1. Замени `off_topic = False` на условие, по которому вопрос считается посторонним. Текст вопроса — в параметре `text`; приводи его к нижнему регистру через `str(text).lower()`, потому что строкой он бывает не всегда.
2. Условие пишется обычным питоном, модель для этого звать не нужно.
3. Прогони список и запусти `ПРОВЕРКА`: нужно отсечь все четыре посторонних и не зарубить ни одного из шести нормальных. Жадное условие вроде «нет слова Ростов — значит постороннее» проверку не пройдёт.

In [ ]:
# ДАНО: шесть нормальных вопросов и четыре посторонних
import sdk_tasks
from sdk_tasks import CITY_QUESTIONS, OFF_TOPIC, count_places
from agents import (Agent, GuardrailFunctionOutput, InputGuardrailTripwireTriggered,
                    Runner, function_tool, input_guardrail)

print("нормальные:", CITY_QUESTIONS[:2], "...")
print("посторонние:", OFF_TOPIC)

In [ ]:
# ЗАДАЧА 8
@input_guardrail(run_in_parallel=False)
async def city_only(ctx, agent, text):
    # определи, посторонний ли вопрос: текст лежит в text, приводи его к нижнему
    # регистру через str(text).lower() — строкой он бывает не всегда.
    # Это обычный питон, модель для проверки звать не нужно
    off_topic = False                       # замени на своё условие
    return GuardrailFunctionOutput(output_info={"вопрос": text}, tripwire_triggered=off_topic)

agent = Agent(name="Справочная", instructions="Отвечай по базе города.",
              tools=[function_tool(count_places)], input_guardrails=[city_only])

asked, blocked = CITY_QUESTIONS + OFF_TOPIC, []
for question in asked:
    try:
        await Runner.run(agent, question)
        blocked.append(False)
    except InputGuardrailTripwireTriggered:
        blocked.append(True)
    print("отсечён " if blocked[-1] else "пропущен", question)

In [ ]:
# ПРОВЕРКА
sdk_tasks.check_guardrail(blocked, asked)

## Задача 9. Триаж · 12 минут

Два готовых агента: `Reference` отвечает фактами по базе, `Analyst` выбирает место под новую точку. Вопросы приходят вперемешку. Нужен третий агент, который ничего не отвечает сам, а раздаёт работу.

Передача через `handoffs` — это обычные инструменты с именами `transfer_to_…`. Отсюда и латиница в именах агентов: имя инструмента в схеме должно быть из букв, цифр и подчёркиваний, иначе SDK переименует его сам и предупредит об этом.

Пустая инструкция триажа выглядит как «модель тупит»: он начинает отвечать сам вместо передачи.

**Что сделать.**

1. Напиши `instructions` триажу. В них должно быть три вещи: что он сам не отвечает никогда; по какому признаку вопрос уходит агенту `Reference`; по какому — агенту `Analyst`. Имена агентов называй так, как они названы в ячейке `ДАНО`.
2. Полезно перечислить примеры формулировок: «сколько чего в городе» — в справочную, «где открыть точку» — аналитику.
3. Прогони и запусти `ПРОВЕРКА`: все шесть вопросов должны уйти по адресу.

In [ ]:
# ДАНО: два агента и шесть вопросов с известным адресатом
import sdk_tasks
from sdk_tasks import MIXED, best_cells, competitors_near, count_places, demand_in_cell, find_places
from agents import Agent, Runner, function_tool

reference = Agent(name="Reference",
                  handoff_description="факты по базе: сколько чего в городе, какие места есть "
                                      "на улице, какие сети представлены",
                  instructions="Отвечай числами по базе города.",
                  tools=[function_tool(count_places), function_tool(find_places)])

analyst = Agent(name="Analyst",
                handoff_description="выбор адреса под новый пункт выдачи: спрос, конкуренты, "
                                    "сравнение кварталов",
                instructions="Оценивай места под новый пункт выдачи по спросу и конкурентам.",
                tools=[function_tool(best_cells), function_tool(demand_in_cell),
                       function_tool(competitors_near)])

for question, want in MIXED[:3]:
    print(f"  {question!r} → {want}")

In [ ]:
# ЗАДАЧА 9
triage = Agent(
    name="Триаж",
    # напиши в instructions три вещи: что триаж сам не отвечает никогда;
    # какие вопросы уходят агенту Reference; какие — агенту Analyst.
    # Имена агентов пиши так, как они названы в ячейке ДАНО
    instructions="...",
    handoffs=[reference, analyst])

routed = []
for question, _ in MIXED:
    routed.append((await Runner.run(triage, question)).last_agent.name)
print(routed)

In [ ]:
# ПРОВЕРКА
sdk_tasks.check_triage(routed)

## Задача 10. Три адреса · 15 минут

Финал. Директор просит три адреса под новые пункты выдачи. Агент отдаёт отчёт объектом — с координатами, спросом и расстоянием до ближайшего конкурента.

`output_type=Answer` — обещание формы: SDK требует от модели ответ по схеме этого класса и разбирает его в объект, поэтому дальше можно писать `report.places[0].lat`, а не резать текст. Поле `checked_cells` — сколько кварталов агент, по его словам, просмотрел, прежде чем выбрать три.

Числа в отчёте выглядят убедительно — и это ничего не говорит о том, откуда они взялись. Единственный способ узнать: пересчитать первое место самому и сравнить. Сойдётся или нет — заранее неизвестно, в этом и смысл сверки.

**Что сделать.**

1. Возьми первое место из отчёта — `report.places[0]`.
2. Посчитай по базе спрос вокруг него: `demand_in_cell(first.lat, first.lon)`.
3. Посчитай конкурентов: `competitors_near(first.lat, first.lon)`.
4. Запусти `ПРОВЕРКА`. Она проверяет твой расчёт по базе, а рядом печатает, сошёлся ли с ним отчёт агента. Расхождение — не ошибка твоего кода: это и есть ответ на вопрос задачи.

In [ ]:
# ДАНО: инструменты кейса и вопрос директора
import sdk_tasks
from sdk_tasks import best_cells, competitors_near, demand_in_cell
from agents import Agent, Runner, function_tool
from pydantic import BaseModel

LETTER = ("Нужны три адреса под новые пункты выдачи в Ростове. "
          "По каждому: координаты, оценка спроса и расстояние до ближайшего конкурента.")

class Spot(BaseModel):
    lat: float
    lon: float
    demand: int
    competitor_distance_m: int
    why: str

class Answer(BaseModel):
    places: list[Spot]
    checked_cells: int

agent = Agent(name="Аналитик",
              instructions="Ищи места по базе города: сначала посмотри кварталы с наибольшим "
                           "спросом, потом проверь по ним спрос и конкурентов. Все числа в "
                           "отчёте бери из ответов инструментов.",
              tools=[function_tool(best_cells), function_tool(demand_in_cell),
                     function_tool(competitors_near)],
              output_type=Answer)

report = (await Runner.run(agent, LETTER)).final_output
for spot in report.places:
    print(spot)

In [ ]:
# ЗАДАЧА 10
first = report.places[0]
hand_demand = ...        # впиши: спрос вокруг first по базе, считает demand_in_cell
hand_competitors = ...   # впиши: конкуренты вокруг first, считает competitors_near

print("агент :", first.demand, "спроса,", first.competitor_distance_m, "м до конкурента")
print("расчёт:", hand_demand, "спроса,", hand_competitors, "конкурентов в радиусе 500 м")

In [ ]:
# ПРОВЕРКА
sdk_tasks.check_report(report, hand_demand, hand_competitors)

## Что дальше

Зачтено шесть задач из десяти — практика сдана. Незачтённые доделываются дома и идут в счёт задания на две недели, вместе с тем, чего в паре не было:

- агент как инструмент (`agent.as_tool`) против передачи `handoffs` — со сравнением на одних и тех же вопросах;
- судья: второй агент проверяет отчёт первого и отправляет на переделку;
- параллельные прогоны: три формулировки сразу, выбор лучшей;
- подтверждение перед дорогим действием;
- свой MCP-сервер с инструментами города.

# Task
Update the code cell `a004` to configure OpenRouter with the "google/gemini-2.5-flash" model and `BASE_URL = "https://openrouter.ai/api/v1"`, and execute the workspace setup (cells `a002` and `a004`) to establish a working environment for the agent development tasks.

## Обновление ячейки a004 под OpenRouter

### Subtask:
Отредактировать ячейку a004 для настройки SDK на использование OpenRouter с моделью google/gemini-2.5-flash.
